# Torsion Scan Guardian — Colab runbook

Zero-cost end-to-end execution of the Phase 0–5 pipeline on Google Colab's free T4 GPU.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Each cell is independent; you can re-run from any point after the install cell finishes.

## 1. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Torch version:', torch.__version__)

## 2. Get the project source

Two options:
- **A.** Mount Google Drive and clone the repo there (preferred — outputs persist across sessions).
- **B.** Clone directly into Colab's ephemeral storage (faster, but you lose results when the session ends).

Set `REPO_URL` to your Git remote, then run **one** of the two cells below.

In [ ]:
# --- Option A: persistent (recommended) ---
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive
REPO_URL = 'https://github.com/YOUR_USERNAME/torsion-scan-guardian.git'  # <-- EDIT
!test -d torsion-scan-guardian || git clone $REPO_URL torsion-scan-guardian
%cd torsion-scan-guardian

In [ ]:
# --- Option B: ephemeral ---
# %cd /content
# REPO_URL = 'https://github.com/YOUR_USERNAME/torsion-scan-guardian.git'  # <-- EDIT
# !git clone $REPO_URL
# %cd torsion-scan-guardian

## 3. Install Python dependencies

Colab already ships PyTorch with CUDA. We pip-install everything else.

**~3–4 minutes the first time, then cached for subsequent restarts in the same session.**

In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q \
    mace-torch \
    ase \
    rdkit \
    matplotlib \
    pandas \
    pydantic \
    pyyaml \
    tqdm \
    pytest
pip install -q -e .

## 4. Install xtb (for the GFN-FF oracle)

Colab doesn't have `xtb` by default. We install via `conda-forge` using `condacolab` (the standard Colab pattern).

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()  # NOTE: this restarts the kernel; re-run cells 1–3 after it finishes.

In [ ]:
# After kernel restart, install xtb itself.
!conda install -y -c conda-forge xtb 2>&1 | tail -5
!which xtb && xtb --version 2>&1 | head -5

## 5. Pre-download MACE-OFF small

One-shot ~7 MB fetch; cached at `~/.cache/mace/`.

In [ ]:
from mace.calculators import mace_off
calc = mace_off(model='small', device='cuda' if torch.cuda.is_available() else 'cpu')
print('MACE-OFF small loaded on', calc.device if hasattr(calc, 'device') else 'unknown')

## 6. Run unit tests (smoke check)

In [ ]:
!python -m pytest -q tests/ -k 'not test_ensemble_predict_h2o'

## 7. Phase 5 demo — sulfanilamide AL run with cloud + safeguard

Same configuration as REPORT §12.5, now on GPU. Expected wall time: **~3–5 minutes** (vs 16 min on CPU).

In [ ]:
%%bash
# Step 7a: build the seed dataset (~3 min).
python scripts/build_seed_dataset.py \
    --smiles 'Nc1ccc(S(=O)(=O)N)cc1' \
    --out data/seed/sulfanilamide_seed.xyz

In [ ]:
%%bash
# Step 7b: fine-tune 3 ensemble members on GPU (~1–2 min total).
for SEED in 0 1 2; do
  python scripts/finetune_member.py \
      --seed $SEED --epochs 5 --lr 5e-4 \
      --train-file data/seed/sulfanilamide_seed.xyz \
      --out-root runs/finetune_sulf \
      --device cuda
done
ls runs/finetune_sulf/*/checkpoints/*.model

In [ ]:
%%bash
# Step 7c: run the 5-cycle AL loop with cloud acquisitions + safeguarded fine-tune.
python -m guardian.cli \
    --config config/default.yaml \
    --smiles 'Nc1ccc(S(=O)(=O)N)cc1' \
    --steps 4000 --temperature 300 --threshold 0.05 \
    --max-triggers 5 --cooldown-steps 200 \
    --cloud-size 5 --cloud-jitter 0.05 \
    --ft-regression-tol 0.10 \
    --checkpoints runs/finetune_sulf/member_seed0/checkpoints/*.model \
                  runs/finetune_sulf/member_seed1/checkpoints/*.model \
                  runs/finetune_sulf/member_seed2/checkpoints/*.model \
    --online-finetune \
    --seed-data-file data/seed/sulfanilamide_seed.xyz \
    --finetune-epochs 2 --finetune-lr 1e-4 \
    --run-dir runs/sulf_phase5_colab

In [ ]:
# Step 7d: analysis figures.
!python scripts/analyse_al_demo.py runs/sulf_phase5_colab

In [ ]:
# Display the timeline figure inline.
from IPython.display import Image
Image('figures/sulf_phase5_colab_timeline.png')

## 8. (Optional) Phase 6 first attempt — glycine zwitterion

Same machinery, a tougher molecule. The §12.7 candidate.

**Note:** charged species like glycine zwitterion may need a `--chrg` flag added to the xtb wrapper (see REPORT §12.7 / time estimate). Run as a probe first.

In [ ]:
# Smoke check: does the SMILES parse and embed?
from rdkit import Chem
from rdkit.Chem import AllChem
mol = Chem.MolFromSmiles('[NH3+]CC(=O)[O-]')
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=0)
print('Atoms:', mol.GetNumAtoms(), '  formal charge sum:', sum(a.GetFormalCharge() for a in mol.GetAtoms()))

## 9. Persist results back to Drive

If you used **Option A** (Drive-mounted clone), `runs/` and `figures/` are already on Drive. If you used Option B, copy them over now:

In [ ]:
# Uncomment if running ephemerally (Option B):
# !cp -r runs /content/drive/MyDrive/torsion_guardian_runs_$(date +%Y%m%d_%H%M)
# !cp -r figures /content/drive/MyDrive/torsion_guardian_figures_$(date +%Y%m%d_%H%M)